In [46]:
import pandas as pd
import numpy as np
from pypfopt import expected_returns, risk_models
from pypfopt.efficient_frontier import EfficientFrontier

In [47]:
df = pd.read_csv("all_sector_data.csv", parse_dates=["Date"], index_col="Date")
#df = df.loc["2024-01-03":"2025-02-28"]
print(df)
# Keep only the 11 S&P 500 sector columns
sector_cols = ["XLC","XLY","XLP","XLE","XLF","XLV","XLI","XLK","XLB","XLRE","XLU"]
df_sectors = df[sector_cols].dropna()

                   XLC         XLY        XLP        XLE        XLF  \
Date                                                                  
2018-06-19   47.037537  104.941498  42.843658  55.261799  24.034578   
2018-06-20   47.621277  105.439095  42.885620  55.505779  23.973063   
2018-06-21   47.329411  104.688042  42.969532  54.478043  23.902761   
2018-06-22   47.536526  104.509666  43.321941  55.564938  23.788515   
2018-06-25   46.557362  102.237732  43.540104  54.448460  23.533672   
...                ...         ...        ...        ...        ...   
2025-02-24  103.220001  217.570007  82.430000  90.949997  50.970001   
2025-02-25  101.690002  216.429993  83.599998  89.639999  50.959999   
2025-02-26  101.510002  215.539993  82.000000  89.129997  50.849998   
2025-02-27  100.529999  212.429993  81.989998  89.610001  51.130001   
2025-02-28  102.000000  215.960007  83.080002  91.000000  52.180000   

                   XLV         XLI         XLK        XLB       XLRE  \
Date

In [48]:
# 'mean_historical_return' 
mu = expected_returns.mean_historical_return(df_sectors, frequency=252)
S = risk_models.sample_cov(df_sectors, frequency=252)

In [49]:
ef = EfficientFrontier(mu, S, weight_bounds=(0,1))
weights_max_sharpe = ef.max_sharpe(risk_free_rate=0.03)
cleaned_weights = ef.clean_weights()

print("Maximum Sharpe Ratio Portfolio Weights:")
for sector, weight in cleaned_weights.items():
    print(f"{sector}: {weight:.2%}")

Maximum Sharpe Ratio Portfolio Weights:
XLC: 0.00%
XLY: 0.00%
XLP: 30.99%
XLE: 0.00%
XLF: 0.00%
XLV: 0.00%
XLI: 0.00%
XLK: 69.01%
XLB: 0.00%
XLRE: 0.00%
XLU: 0.00%


In [51]:
daily_returns = df_sectors.pct_change().dropna()
weights_series = pd.Series(weights_max_sharpe)

# Compute portfolio daily returns
portfolio_returns = daily_returns.dot(weights_series)

# Compute cumulative returns
cumulative_returns = (1 + portfolio_returns).cumprod()

# Compute running maximum
running_max = cumulative_returns.cummax()

# Compute drawdown
drawdown = (cumulative_returns - running_max) / running_max

# Compute maximum drawdown
max_drawdown = drawdown.min()

print(f"Maximum Drawdown: {max_drawdown:.2%}")


Maximum Drawdown: -28.94%
